In [39]:
import pandas as pd
import plotly.express as px
import folium
import yaml
import numpy as np
import geopandas as gpd
from pyproj import Transformer
from shapely.geometry import Polygon
from folium import plugins
import requests

import calliope

# We increase logging verbosity
calliope.set_log_verbosity("INFO", include_solver_output=False)

In [40]:
model = calliope.read_yaml('model.yaml')

[2026-03-23 10:03:07] INFO     Math init | loading pre-defined math.
[2026-03-23 10:03:07] INFO     Math init | loading math files {'operate', 'spores', 'milp', 'storage_inter_cluster', 'base'}.
[2026-03-23 10:03:07] INFO     Model: preprocessing data
[2026-03-23 10:03:07] INFO     Math build | building applied math with ['base'].
[2026-03-23 10:03:08] INFO     input data `color` not defined in model math; it will not be available in the optimisation problem.
[2026-03-23 10:03:08] INFO     input data `name` not defined in model math; it will not be available in the optimisation problem.
[2026-03-23 10:03:08] INFO     input data `flow_cap` not defined in model math; it will not be available in the optimisation problem.
[2026-03-23 10:03:08] INFO     input data `storage_cap` not defined in model math; it will not be available in the optimisation problem.
[2026-03-23 10:03:08] INFO     input data `link_to` not defined in model math; it will not be available in the optimisation problem.
[2

In [41]:
model.inputs

<xarray.Dataset> Size: 85kB
Dimensions:                     (costs: 1, techs: 15, nodes: 12, carriers: 1,
                                 timesteps: 48)
Coordinates:
  * costs                       (costs) object 8B 'monetary'
  * techs                       (techs) object 120B 'HV_station_to_transmissi...
  * carriers                    (carriers) object 8B 'electricity'
  * nodes                       (nodes) object 96B 'HV_station' ... 'transmis...
  * timesteps                   (timesteps) datetime64[ns] 384B 2024-04-01 .....
Data variables: (12/35)
    cost_interest_rate          (costs) float64 8B 0.1
    bigM                        float64 8B 1e+06
    objective_cost_weights      (costs) float64 8B 1.0
    base_tech                   (techs) object 120B 'transmission' ... 'trans...
    carrier_in                  (nodes, techs, carriers) bool 180B True ... True
    color                       (techs) object 120B '#823739' ... '#823739'
    ...                          ...
    cost_flow_out               (costs, timesteps, techs) float64 6kB nan ......
    sink_use_equals             (timesteps, techs, nodes) float64 69kB nan .....
    definition_matrix           (nodes, techs, carriers) bool 180B True ... True
    distance                    (techs) float64 120B 0.05581 nan ... 0.5484
    timestep_resolution         (timesteps) float64 384B 1.0 1.0 1.0 ... 1.0 1.0
    timestep_weights            (timesteps) float64 384B 1.0 1.0 1.0 ... 1.0 1.0

In [42]:
model.inputs.flow_cap_max.to_series().dropna()  

techs
HV_station_to_transmission_1            100000.0
battery                                  50000.0
pv                                        1000.0
supply_grid_power                        50000.0
transmission_1_to_transmission_2        100000.0
transmission_1_to_transmission_3        100000.0
transmission_2_to_data_center_3         100000.0
transmission_3_to_data_center_2         100000.0
transmission_3_to_transmission_4        100000.0
transmission_3_to_transmission_5        100000.0
transmission_4_to_data_center_1         100000.0
transmission_5_to_transmission_6        100000.0
transmission_6_to_transmission_7        100000.0
transmission_7_to_residential_demand    100000.0
Name: flow_cap_max, dtype: float64

In [43]:
model.inputs.sink_use_equals.sum(
    "timesteps", min_count=1, skipna=True
).to_series().dropna()

techs               nodes             
demand_electricity  data_center_1         240000.000000
                    data_center_2         384000.000000
                    data_center_3         576000.000000
                    residential_demand     31895.209428
Name: sink_use_equals, dtype: float64

In [44]:
model.build(force=True)

[2026-03-23 10:03:08] INFO     Model: backend build starting
[2026-03-23 10:03:09] INFO     Optimisation Model | parameters/lookups | Generated.
[2026-03-23 10:03:09] INFO     Optimisation Model | variables | Generated.
[2026-03-23 10:03:10] INFO     Optimisation Model | global_expressions | Generated.
[2026-03-23 10:03:11] INFO     Optimisation Model | constraints | Generated.
[2026-03-23 10:03:11] INFO     Optimisation Model | piecewise_constraints | Generated.
[2026-03-23 10:03:11] INFO     Optimisation Model | objectives | Generated.
[2026-03-23 10:03:11] INFO     Model: backend build complete


In [45]:
model.backend.parameters

<xarray.Dataset> Size: 84kB
Dimensions:                             (costs: 1, techs: 15, timesteps: 48,
                                         nodes: 12)
Coordinates:
  * costs                               (costs) object 8B 'monetary'
  * techs                               (techs) object 120B 'HV_station_to_tr...
  * timesteps                           (timesteps) datetime64[ns] 384B 2024-...
  * nodes                               (nodes) object 96B 'HV_station' ... '...
Data variables: (12/59)
    area_use_max                        float64 8B nan
    area_use_min                        float64 8B nan
    area_use_per_flow_cap               float64 8B nan
    available_area                      float64 8B nan
    bigM                                object 8B parameters[bigM][0]
    cost_flow_cap_per_distance          (costs, techs) object 120B parameters...
    ...                                  ...
    storage_cap_per_unit                float64 8B nan
    storage_discharge_depth             float64 8B nan
    storage_initial                     float64 8B nan
    storage_loss                        (techs) object 120B nan ... nan
    timestep_resolution                 (timesteps) object 384B parameters[ti...
    timestep_weights                    (timesteps) object 384B parameters[ti...

In [46]:
model.solve(solver='gurobi')

[2026-03-23 10:03:12] INFO     Optimisation model | starting model in base mode.
[2026-03-23 10:03:12] INFO     Backend: solver finished running. Time since start of solving optimisation problem: 0:00:00.358301
[2026-03-23 10:03:12] INFO     Postprocessing: applied zero threshold 1e-10 to model results.
[2026-03-23 10:03:12] INFO     Postprocessing: ended. Time since start of solving optimisation problem: 0:00:00.417628
[2026-03-23 10:03:12] INFO     Backend: model solve completed. Time since start of solving optimisation problem: 0:00:00.418630


In [47]:
model.backend.parameters

<xarray.Dataset> Size: 84kB
Dimensions:                             (costs: 1, techs: 15, timesteps: 48,
                                         nodes: 12)
Coordinates:
  * costs                               (costs) object 8B 'monetary'
  * techs                               (techs) object 120B 'HV_station_to_tr...
  * timesteps                           (timesteps) datetime64[ns] 384B 2024-...
  * nodes                               (nodes) object 96B 'HV_station' ... '...
Data variables: (12/59)
    area_use_max                        float64 8B nan
    area_use_min                        float64 8B nan
    area_use_per_flow_cap               float64 8B nan
    available_area                      float64 8B nan
    bigM                                object 8B parameters[bigM][0]
    cost_flow_cap_per_distance          (costs, techs) object 120B parameters...
    ...                                  ...
    storage_cap_per_unit                float64 8B nan
    storage_discharge_depth             float64 8B nan
    storage_initial                     float64 8B nan
    storage_loss                        (techs) object 120B nan ... nan
    timestep_resolution                 (timesteps) object 384B parameters[ti...
    timestep_weights                    (timesteps) object 384B parameters[ti...

In [48]:
model.results

<xarray.Dataset> Size: 647kB
Dimensions:                     (nodes: 12, techs: 15, carriers: 1,
                                 timesteps: 48, costs: 1)
Coordinates:
  * techs                       (techs) object 120B 'HV_station_to_transmissi...
  * nodes                       (nodes) object 96B 'HV_station' ... 'transmis...
  * carriers                    (carriers) object 8B 'electricity'
  * timesteps                   (timesteps) datetime64[ns] 384B 2024-04-01 .....
  * costs                       (costs) object 8B 'monetary'
Data variables: (12/24)
    flow_cap                    (nodes, techs, carriers) float64 1kB 2.567e+0...
    link_flow_cap               (techs) float64 120B 2.567e+04 nan ... 1e+05
    flow_out                    (nodes, techs, carriers, timesteps) float64 69kB ...
    flow_in                     (nodes, techs, carriers, timesteps) float64 69kB ...
    flow_export                 (nodes, techs, carriers, timesteps) float64 69kB ...
    source_use                  (nodes, techs, timesteps) float64 69kB nan .....
    ...                          ...
    min_cost_optimisation       float64 8B 6.796e+08
    capacity_factor             (nodes, techs, carriers, timesteps) float64 69kB ...
    systemwide_capacity_factor  (techs, carriers) float64 120B 0.4981 ... 0.0...
    systemwide_levelised_cost   (techs, costs, carriers) float64 120B 75.11 ....
    total_levelised_cost        (costs, carriers) float64 8B 549.1
    unmet_sum                   (nodes, carriers, timesteps) float64 5kB 0.0 ...

In [49]:
costs = model.results.cost.to_series().dropna()
costs.head()

nodes          techs                            costs   
HV_station     HV_station_to_transmission_1     monetary    4.610073e+07
               supply_grid_power                monetary    5.174831e+06
data_center_1  battery                          monetary    1.211889e-01
               pv                               monetary    6.036607e-03
               transmission_4_to_data_center_1  monetary    3.778915e+07
Name: cost, dtype: float64

In [50]:
lcoes = (
    model.results.systemwide_levelised_cost.sel(carriers="electricity")
    .to_series()
    .dropna()
)
lcoes.head()

techs                             costs   
HV_station_to_transmission_1      monetary    75.112534
battery                           monetary     0.000032
pv                                monetary     0.000002
supply_grid_power                 monetary     4.213346
transmission_1_to_transmission_2  monetary    39.149836
Name: systemwide_levelised_cost, dtype: float64

In [51]:
# We set the color mapping to use in all our plots by extracting the colors defined in the technology definitions of our model.
colors = model.inputs.color.to_series().to_dict()

In [52]:
df_electricity = (
    (model.results.flow_out.fillna(0) - model.results.flow_in.fillna(0))
    .sel(carriers="electricity")
    .sum("nodes")
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow in/out (kWh)")
    .reset_index()
)
df_electricity_demand = df_electricity[df_electricity.techs == "demand_electricity"]
df_electricity_other = df_electricity[df_electricity.techs != "demand_electricity"]

print(df_electricity.head())

fig1 = px.bar(
    df_electricity_other,
    x="timesteps",
    y="Flow in/out (kWh)",
    color="techs",
    color_discrete_map=colors,
)
fig1.add_scatter(
    x=df_electricity_demand.timesteps,
    y=-1 * df_electricity_demand["Flow in/out (kWh)"],
    marker_color="black",
    name="demand",
)

                          techs           timesteps  Flow in/out (kWh)
0  HV_station_to_transmission_1 2024-04-01 00:00:00         -14.380761
1  HV_station_to_transmission_1 2024-04-01 01:00:00         -14.383718
2  HV_station_to_transmission_1 2024-04-01 02:00:00         -14.385000
3  HV_station_to_transmission_1 2024-04-01 03:00:00         -14.385000
4  HV_station_to_transmission_1 2024-04-01 04:00:00         -14.384953


In [53]:
carriers = ["electricity"]
df_flows = (
    (model.results.flow_out.fillna(0) - model.results.flow_in.fillna(0))
    .sel(carriers=carriers)
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow in/out (kWh)")
    .reset_index()
)
df_demand = df_flows[df_flows.techs.str.contains("demand")]
df_flows_other = df_flows[~df_flows.techs.str.contains("demand")]

print(df_flows.head())

node_order = df_flows_other.nodes.unique()

fig = px.bar(
    df_flows_other,
    x="timesteps",
    y="Flow in/out (kWh)",
    facet_row="nodes",
    facet_col="carriers",
    color="techs",
    category_orders={"nodes": node_order, "carriers": carriers},
    height=1000,
    color_discrete_map=colors,
)

showlegend = True
# we reverse the node order (`[::-1]`) because the rows are numbered from bottom to top.
for row, node in enumerate(node_order[::-1]):
    for col, carrier in enumerate(carriers):
        demand_ = df_demand.loc[
            (df_demand.nodes == node) & (df_demand.techs == f"demand_{carrier}"),
            "Flow in/out (kWh)",
        ]
        if not demand_.empty:
            fig.add_scatter(
                x=model.results.timesteps.values,
                y=-1 * demand_,
                row=row + 1,
                col=col + 1,
                marker_color="black",
                name="Demand",
                legendgroup="demand",
                showlegend=showlegend,
            )
            showlegend = False
fig.update_yaxes(matches=None)
fig.show()

        nodes                         techs     carriers           timesteps  \
0  HV_station  HV_station_to_transmission_1  electricity 2024-04-01 00:00:00   
1  HV_station  HV_station_to_transmission_1  electricity 2024-04-01 01:00:00   
2  HV_station  HV_station_to_transmission_1  electricity 2024-04-01 02:00:00   
3  HV_station  HV_station_to_transmission_1  electricity 2024-04-01 03:00:00   
4  HV_station  HV_station_to_transmission_1  electricity 2024-04-01 04:00:00   

   Flow in/out (kWh)  
0      -25646.421508  
1      -25651.694770  
2      -25653.981968  
3      -25653.981968  
4      -25653.897259  


In [54]:
df_capacity = (
    model.results.flow_cap.where(
        ~model.inputs.base_tech.str.contains("demand|transmission")
    )
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow capacity (kW)")
    .reset_index()
)

print(df_capacity.head())

fig = px.bar(
    df_capacity,
    x="nodes",
    y="Flow capacity (kW)",
    color="techs",
    facet_col="carriers",
    color_discrete_map=colors,
)
fig.show()

           nodes              techs     carriers  Flow capacity (kW)
0     HV_station  supply_grid_power  electricity        25669.404330
1  data_center_1            battery  electricity         2007.566125
2  data_center_1                 pv  electricity         1000.000000
3  data_center_2            battery  electricity          672.779272
4  data_center_2                 pv  electricity         1000.000000


In [55]:
with open("model.yaml", "r", encoding="utf-8") as f:
    model_def = yaml.safe_load(f)

node_techs = {
    node: list((node_data.get("techs") or {}).keys())
    for node, node_data in model_def.get("nodes", {}).items()
}

In [56]:
# Build a simple system map (nodes + links)
nodes = pd.read_csv("nodes_coordinates.csv")
links = pd.read_csv("links_techs.csv")

flow_cap = (
    model.results.flow_cap.to_series().dropna()
    .to_frame("Flow capacity (kW)")
    .reset_index()
)
flow_cap_lookup = dict(zip(flow_cap["techs"], flow_cap["Flow capacity (kW)"]))

center = [nodes.latitude.mean(), nodes.longitude.mean()]
system_map = folium.Map(location=center, zoom_start=15, tiles="CartoDB voyager")

# Add link lines
for _, row in links.iterrows():
    from_row = nodes.loc[nodes.nodes == row["link_from"]].iloc[0]
    to_row = nodes.loc[nodes.nodes == row["link_to"]].iloc[0]
    capacity = flow_cap_lookup.get(row["techs"], row.get("flow_cap_max"))
    popup = (
        f"<b>{row['techs']}</b><br>From: {row['link_from']}<br>To: {row['link_to']}"
    )
    if capacity is not None:
        popup += f"<br>Capacity: {capacity:.2f} kW"
    folium.PolyLine(
        locations=[[from_row.latitude, from_row.longitude], [to_row.latitude, to_row.longitude]],
        color=row.get("color", "#1f77b4"),
        weight=3,
        opacity=1,
        popup=popup,
    ).add_to(system_map)

# Add node markers
color_map = model.inputs.color.to_series().to_dict()
for _, row in nodes.iterrows():
    node = row["nodes"]
    techs = node_techs.get(node, [])
    base_types = (
        model.inputs.base_tech.sel(techs=techs).to_series().to_dict()
        if techs
        else {}
    )

    node_type = "Other"
    if any(t == "demand" for t in base_types.values()):
        node_type = "Demand"
    elif any(t == "supply" for t in base_types.values()):
        node_type = "Supply"

    marker_color = "#666666"
    for tech in techs:
        if tech in color_map:
            marker_color = color_map[tech]
            break

    popup = f"<b>{node}</b> ({node_type})<br>Techs: {', '.join(techs)}"
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=6,
        color=marker_color,
        fill=True,
        fillColor=marker_color,
        fillOpacity=1,
        popup=popup,
    ).add_to(system_map)

system_map.save("system_map.html")

In [ ]:
# Export Comprehensive Bill of Materials (All Technologies + Transmission)
from math import radians, cos, sin, asin, sqrt

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate the great circle distance between two points on earth"""
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371000  # Radius of earth in meters
    return c * r

# ===== Non-transmission Technologies (Supply, Storage, etc.) =====
non_transmission_bom = (
    model.results.flow_cap
    .where(~model.inputs.base_tech.isin(["demand", "transmission"]))
    .to_series()
    .where(lambda x: x > 0)
    .dropna()
    .to_frame("capacity_kw")
    .reset_index()
)
non_transmission_bom = non_transmission_bom.rename(columns={"techs": "technology"})
non_transmission_bom["distance_m"] = None

# ===== Transmission Technologies (Power Lines) =====
power_lines_cap = (
    model.results.flow_cap
    .where(model.inputs.base_tech == "transmission")
    .to_series()
    .where(lambda x: x > 0)
    .dropna()
    .to_frame("capacity_kw")
    .reset_index()
)
power_lines_cap = power_lines_cap.groupby("techs", as_index=False)["capacity_kw"].first()

# Load links and coordinates to calculate distances
links_data = pd.read_csv("links_techs.csv")
nodes_coords = pd.read_csv("nodes_coordinates.csv")
coord_dict = dict(zip(nodes_coords['nodes'], zip(nodes_coords['latitude'], nodes_coords['longitude'])))

# Calculate distances
links_data['distance_m'] = links_data.apply(
    lambda row: haversine_distance(
        coord_dict[row['link_from']][0], coord_dict[row['link_from']][1],
        coord_dict[row['link_to']][0], coord_dict[row['link_to']][1]
    ),
    axis=1
)

# Merge transmission capacity with distances
transmission_bom = power_lines_cap.merge(
    links_data[["techs", "distance_m"]],
    left_on="techs",
    right_on="techs",
    how="left"
)
transmission_bom = transmission_bom.rename(columns={"techs": "technology"})

# ===== Combine All Technologies =====
# Explicitly ensure distance_m is float64 before concatenating to avoid FutureWarning
non_transmission_bom['distance_m'] = non_transmission_bom['distance_m'].astype('float64')
transmission_bom['distance_m'] = transmission_bom['distance_m'].astype('float64')

bom_df = pd.concat([non_transmission_bom, transmission_bom], ignore_index=True)

# Sort by technology name and capacity
bom_df = bom_df.sort_values(
    by=["technology", "capacity_kw"],
    ascending=[True, False]
).reset_index(drop=True)

# Save to single comprehensive CSV
os.makedirs("outputs", exist_ok=True)
bom_df.to_csv("outputs/bill_of_materials.csv", index=False)
#print(f"\nSaved to outputs/bill_of_materials.csv")


Saved to outputs/bill_of_materials.csv
